# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import pandas as pd

file_path = r"../data/raw/hpi_at_metro.csv"

In [17]:
columns = [
    "metro",
    "cbsa",
    "year",
    "quarter",
    "hpi",
    "standard_error"
]

In [3]:
df = pd.read_csv(
    file_path,
    header=None,
    names=columns,
    na_values="-"
)

In [4]:
print(df.head())
print(df.shape)
print(df.dtypes)
df.info()

         metro   cbsa  year  quarter  hpi annual_change
0  Abilene, TX  10180  1975        1  NaN           NaN
1  Abilene, TX  10180  1975        2  NaN           NaN
2  Abilene, TX  10180  1975        3  NaN           NaN
3  Abilene, TX  10180  1975        4  NaN           NaN
4  Abilene, TX  10180  1976        1  NaN           NaN
(84460, 6)
metro             object
cbsa               int64
year               int64
quarter            int64
hpi              float64
annual_change     object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84460 entries, 0 to 84459
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   metro          84460 non-null  object 
 1   cbsa           84460 non-null  int64  
 2   year           84460 non-null  int64  
 3   quarter        84460 non-null  int64  
 4   hpi            71072 non-null  float64
 5   annual_change  71072 non-null  object 
dtypes: float64(1), int64(3),

In [5]:
charleston = df[
    (df["cbsa"] == 16700) &
    (df["year"] >= 2019)
].copy()

print(charleston)

                                 metro   cbsa  year  quarter     hpi  \
13772  Charleston-North Charleston, SC  16700  2019        1  328.29   
13773  Charleston-North Charleston, SC  16700  2019        2  333.75   
13774  Charleston-North Charleston, SC  16700  2019        3  337.26   
13775  Charleston-North Charleston, SC  16700  2019        4  339.05   
13776  Charleston-North Charleston, SC  16700  2020        1  345.17   
13777  Charleston-North Charleston, SC  16700  2020        2  347.76   
13778  Charleston-North Charleston, SC  16700  2020        3  355.36   
13779  Charleston-North Charleston, SC  16700  2020        4  361.06   
13780  Charleston-North Charleston, SC  16700  2021        1  371.03   
13781  Charleston-North Charleston, SC  16700  2021        2  394.58   
13782  Charleston-North Charleston, SC  16700  2021        3  417.95   
13783  Charleston-North Charleston, SC  16700  2021        4  435.78   
13784  Charleston-North Charleston, SC  16700  2022        1  45

In [7]:
# Descriptive statistics for FHFA House Price Index
print("\nFHFA HPI Descriptive Statistics")
print(df["hpi"].describe())

print("\nAdditional Statistics")
print("Median:", df["hpi"].median())
print("Variance:", df["hpi"].var())


FHFA HPI Descriptive Statistics
count    71072.000000
mean       162.878121
std         88.734259
min         18.600000
25%        100.000000
50%        145.610000
75%        199.400000
max        670.570000
Name: hpi, dtype: float64

Additional Statistics
Median: 145.61
Variance: 7873.768748142086


In [8]:
print("\nDistribution Shape")
print("Skewness:", df["hpi"].skew())
print("\nMissing HPI Values")
print(df["hpi"].isna().sum())


Distribution Shape
Skewness: 1.3834199213494325

Missing HPI Values
13388


In [10]:
coverage = (
    df.groupby("year")
      .agg(
          total_rows=("hpi", "size"),
          valid_hpi=("hpi", "count")
      )
)

coverage["missing_hpi"] = coverage["total_rows"] - coverage["valid_hpi"]
coverage["coverage_pct"] = (
    coverage["valid_hpi"] / coverage["total_rows"] * 100
).round(2)

print(coverage.loc[2000:2026])

      total_rows  valid_hpi  missing_hpi  coverage_pct
year                                                  
2000        1640       1637            3         99.82
2001        1640       1640            0        100.00
2002        1640       1640            0        100.00
2003        1640       1640            0        100.00
2004        1640       1640            0        100.00
2005        1640       1640            0        100.00
2006        1640       1640            0        100.00
2007        1640       1640            0        100.00
2008        1640       1640            0        100.00
2009        1640       1640            0        100.00
2010        1640       1640            0        100.00
2011        1640       1640            0        100.00
2012        1640       1640            0        100.00
2013        1640       1640            0        100.00
2014        1640       1640            0        100.00
2015        1640       1640            0        100.00
2016      

In [11]:
print("\n2026 HPI Coverage")
print(
    df[df["year"] == 2026]
    .groupby("quarter")
    .agg(
        total_rows=("hpi", "size"),
        valid_hpi=("hpi", "count")
    )
)


2026 HPI Coverage
         total_rows  valid_hpi
quarter                       
1               410        410
2               410        410


In [12]:
print("\nUnique CBSAs:")
print(df["cbsa"].nunique())

print("\nUnique CBSAs from 2005 onward:")
print(df[df["year"] >= 2005]["cbsa"].nunique())


Unique CBSAs:
410

Unique CBSAs from 2005 onward:
410


In [14]:
fhfa_analysis = df[
    (df["year"] >= 2005) &
    (df["year"] <= 2025) &
    (df["hpi"].notna())
].copy()

print("\nFHFA Analysis Period: 2005–2025")
print(fhfa_analysis["hpi"].describe())

print("\nAdditional HPI Statistics")
print("Median:", fhfa_analysis["hpi"].median())
print("Variance:", fhfa_analysis["hpi"].var())
print("Skewness:", fhfa_analysis["hpi"].skew())

print("\nAnalysis Dataset Shape")
print(fhfa_analysis.shape)


FHFA Analysis Period: 2005–2025
count    34439.000000
mean       223.092612
std         79.419182
min        100.290000
25%        166.480000
50%        196.600000
75%        258.415000
max        657.880000
Name: hpi, dtype: float64

Additional HPI Statistics
Median: 196.6
Variance: 6307.406427544046
Skewness: 1.5152561059258942

Analysis Dataset Shape
(34439, 6)


In [15]:
print("\nAnnual Change Sample")
print(df["annual_change"].dropna().head(20))

print("\nAnnual Change Data Type")
print(df["annual_change"].dtype)


Annual Change Sample
45    ( 2.56)
46    ( 2.78)
47    ( 2.84)
48    ( 2.88)
49    ( 2.64)
50    ( 3.07)
51    ( 3.60)
52    ( 3.16)
53    ( 2.53)
54    ( 2.39)
55    ( 2.61)
56    ( 2.66)
57    ( 2.58)
58    ( 2.61)
59    ( 2.89)
60    ( 2.63)
61    ( 2.33)
62    ( 2.48)
63    ( 2.48)
64    ( 2.98)
Name: annual_change, dtype: object

Annual Change Data Type
object


In [16]:
print("\nAnnual Change - First 20")
print(df["annual_change"].dropna().head(20))

print("\nAnnual Change - Values Around 2008")
print(
    df.loc[
        (df["year"] >= 2007) & (df["year"] <= 2009),
        ["metro", "year", "quarter", "hpi", "annual_change"]
    ].head(30)
)


Annual Change - First 20
45    ( 2.56)
46    ( 2.78)
47    ( 2.84)
48    ( 2.88)
49    ( 2.64)
50    ( 3.07)
51    ( 3.60)
52    ( 3.16)
53    ( 2.53)
54    ( 2.39)
55    ( 2.61)
56    ( 2.66)
57    ( 2.58)
58    ( 2.61)
59    ( 2.89)
60    ( 2.63)
61    ( 2.33)
62    ( 2.48)
63    ( 2.48)
64    ( 2.98)
Name: annual_change, dtype: object

Annual Change - Values Around 2008
           metro  year  quarter     hpi annual_change
128  Abilene, TX  2007        1  162.48       ( 3.39)
129  Abilene, TX  2007        2  164.10       ( 3.27)
130  Abilene, TX  2007        3  163.20       ( 3.24)
131  Abilene, TX  2007        4  163.63       ( 3.36)
132  Abilene, TX  2008        1  165.66       ( 3.36)
133  Abilene, TX  2008        2  165.03       ( 3.34)
134  Abilene, TX  2008        3  166.20       ( 3.41)
135  Abilene, TX  2008        4  169.51       ( 3.65)
136  Abilene, TX  2009        1  170.56       ( 3.43)
137  Abilene, TX  2009        2  169.96       ( 3.42)
138  Abilene, TX  2009       